# 45. Four model families this stack has never contained

**One variable against ledger row 93** (`xgb_te_fe`, CV 0.968005): the learner. Same 49
features, same encoder fingerprinted `0642e41750ef8bab`, same ratio block, same folds, same seed.
Four arms, four learners that appear nowhere in the 41 members.

| arm | family | why it is here |
|---|---|---|
| `logit_te_fe` | L2 logistic regression | the stack contains **no linear model at all** |
| `hgb_te_fe` | sklearn HistGradientBoosting | a fourth boosting implementation, different regularisation |
| `et_te_fe` | ExtraTrees | randomised split thresholds, not greedy |
| `rf_te_fe` | RandomForest | bagged greedy trees, no boosting |

## Why this and not more tuning

Measured here on 2026-08-22: the 41-member meta-matrix has **one principal component explaining
95 percent of its variance**, nine explaining 99 percent. Forty-one members, and nearly all of
them are the same three gradient-boosted families over two representations plus two MLPs. The
condition number is fine, so this is not a numerical problem. It is a **diversity** problem.

Everything cheap left in the tuning direction is exhausted and measured: four knob sweeps null,
`max_bin` null, three separate routes to per-value structure null. What has never been tried is a
model that is wrong in a **different shape**.

The public evidence is specific and it is the reason a linear model leads this list. In
srcJ's 177-model coefficient table a plain logistic regression with a solo OOF of **0.9589**
takes coefficient **+0.137**, sixth highest, ahead of eight boosted trees that each score 0.008 to
0.010 better than it. Our own row 96 is the same lesson from our own data: `xgb_raw_fe` sits
0.0033 **below** the stack it joined and fired at 5/5 folds.

## The preprocessing is forced by the learner, and that is a second variable

XGBoost routes NaN natively and takes categoricals directly. Three of these four cannot:

| arm | imputation | scaling | categoricals |
|---|---|---|---|
| `logit_te_fe` | median | standardised | one-hot |
| `et_te_fe`, `rf_te_fe` | median | none | one-hot |
| `hgb_te_fe` | native, none applied | none | one-hot |

Every step is fit **inside the fold** on training rows only. This means each arm differs from row
93 by the learner *and* by whatever preprocessing that learner requires, which is strictly two
changes. It is stated rather than hidden, and it is unavoidable: there is no version of a
logistic regression that consumes a NaN. Row 26 had the same shape when CatBoost first appeared
against row 17 and it was recorded the same way.

## What decides whether this worked

**Not the solo CV.** Three of these four will land well below 0.968005 and that is expected. A
linear model on 49 columns cannot compete with a tuned booster on raw accuracy, and it is not
being asked to.

The question is what a fitted combiner does with them, which is `46_stack_v3.ipynb` and a separate
row. This notebook writes vectors.

## The prediction, written before the run

Solo, in order: `hgb_te_fe` highest at 0.9670 to 0.9678, then `et_te_fe` and `rf_te_fe` around
0.960 to 0.966, then `logit_te_fe` lowest at 0.955 to 0.963.

**In the gate I expect the logistic regression to take the largest coefficient of the four**
despite being the weakest, and I expect at least two of the four to survive. That is the claim
worth being wrong about, and it is the one the public table and row 96 both support.

**The honest case against.** My case-against sections have beaten my predictions three runs
running, so here is the strongest one I can make. Our 49-feature frame already contains 24
smoothed target statistics. A logistic regression on target-encoded columns is close to a
re-weighted version of the encoder itself, so it may be far *less* independent than srcJ's,
which sat on a different feature set. If all four land at near-zero coefficients then this stack
really is saturated in model space as well as in feature space, and the remaining gap to the top
decile is not reachable with own models.

## What this decides

Nothing about the stack. No submission csv.

In [ ]:
SMOKE = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0
TARGET = "addicted_label"
LR = 0.05
N_EST = 2000
MAX_DEPTH = 6
BENCH_EST = 200
PROBE_FOLD = 0
THREADS = -1

ARMS = ["logit_te_fe", "hgb_te_fe", "et_te_fe", "rf_te_fe"]

# Forest sizes chosen for cost, not for accuracy. 691,369 rows by 49 columns is a lot of
# tree to grow; min_samples_leaf keeps them from memorising and keeps the run affordable.
N_TREES = 300
MIN_LEAF = 20

# Every arm changes the learner against row 93, plus whatever preprocessing the learner
# forces. Stated in the header rather than hidden.
BASELINE_NAME, BASELINE_CV, BASELINE_ROW = "xgb_te_fe", 0.968005, "row 93 xgb_te_fe"

MAX_HOURS = 9.0
EXPECTED_FOLD_SHA = "ec282b0968059676"
EXPECTED_ENCODER_FP = "0642e41750ef8bab"
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

print(f"SMOKE = {SMOKE}   arms {ARMS}")
print(f"forests: {N_TREES} trees, min_samples_leaf {MIN_LEAF}")

## Stage 1. Data, folds, leak checklist

In [ ]:
import ast
import gc
import hashlib
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import (ExtraTreesClassifier, HistGradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, BENCH_EST, N_TREES = 200, 50, 40
    THREADS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
X = train[COLS].copy()
X_test = test[COLS].copy()

folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i
sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print(f"\nrows {len(train):,}   fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
print("SMOKE: sha EXPECTED to differ, not a check." if SMOKE else
      ("fold alignment: VERIFIED" if ALIGNED else "fold alignment: MISMATCH"))

## Stage 2. The encoder, fingerprinted, and the ratio block

In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    """Semantic checksum of the encoder functions inside a block of source."""
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))

theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here        : {mine}")
print(f"encoder fingerprint in 13       : {theirs}")
print(f"rows 26 and 33 recorded         : 0642e41750ef8bab")
print("encoder: IDENTICAL to row 17's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - this is NOT one variable")
print()
print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")

In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

In [ ]:
DST, SM, GM, WS = ("daily_screen_time_hours", "social_media_hours",
                   "gaming_hours", "work_study_hours")
SLP, WKD = "sleep_hours", "weekend_screen_time"
NOTIF, OPENS = "notifications_per_day", "app_opens_per_day"

RATIO_COLS = ["component_total", "slack", "weekend_lift", "weekend_ratio",
              "social_share", "gaming_share", "work_share", "sleep_minus_screen",
              "screen_to_sleep", "opens_per_hour", "notif_per_hour",
              "notif_per_open", "engagement"]


def safe_div(a, b):
    b = b.replace(0, np.nan)
    return a / b


def ratio_block(df):
    """Notebook 40's 13 composition features. A pure function of the feature columns."""
    o = pd.DataFrame(index=df.index)
    o["component_total"] = df[[SM, GM, WS]].sum(axis=1, min_count=3)
    o["slack"] = df[DST] - o["component_total"]
    o["weekend_lift"] = df[WKD] - df[DST]
    o["weekend_ratio"] = safe_div(df[WKD], df[DST])
    o["social_share"] = safe_div(df[SM], df[DST])
    o["gaming_share"] = safe_div(df[GM], df[DST])
    o["work_share"] = safe_div(df[WS], df[DST])
    o["sleep_minus_screen"] = df[SLP] - df[DST]
    o["screen_to_sleep"] = safe_div(df[DST], df[SLP])
    o["opens_per_hour"] = safe_div(df[OPENS], df[DST])
    o["notif_per_hour"] = safe_div(df[NOTIF], df[DST])
    o["notif_per_open"] = safe_div(df[NOTIF], df[OPENS])
    o["engagement"] = df[NOTIF] + df[OPENS]
    return o.replace([np.inf, -np.inf], np.nan).astype(np.float64)


rng = np.random.default_rng(0)
_perm = train.copy()
_perm[TARGET] = rng.permutation(train[TARGET].to_numpy())
BLOCK_OK = bool(ratio_block(_perm).equals(ratio_block(train))
                and list(ratio_block(train).columns) == RATIO_COLS)
print(f"ratio block is a pure function of the features, not of y: {BLOCK_OK}")
del _perm
gc.collect()

# One-hot levels, from train and test features only. No target is involved.
DUMMIES = sorted({f"{c}={v}" for c in CAT
                  for v in pd.concat([train[c], test[c]]).dropna().unique()})
print(f"{len(DUMMIES)} one-hot columns from {len(CAT)} categoricals: {DUMMIES}")


def onehot(df):
    o = pd.DataFrame(index=df.index)
    for d in DUMMIES:
        c, v = d.split("=", 1)
        o[d] = (df[c].astype(str) == v).astype(np.float64)
    return o


def build_arm(fold, want_test=False):
    """Row 93's 49-column frame, with categoricals one-hot instead of native."""
    tr = np.where(folds != fold)[0]
    va = np.where(folds == fold)[0]
    Etr, Eva, Ete = build(X, y, tr, va, X_test if want_test else None)
    def assemble(E, src):
        num = E.drop(columns=CAT).reset_index(drop=True)
        return pd.concat([num, onehot(E[CAT]).reset_index(drop=True),
                          ratio_block(src).reset_index(drop=True)], axis=1)
    Xtr = assemble(Etr, train.iloc[tr])
    Xva = assemble(Eva, train.iloc[va])
    Xte = assemble(Ete, test) if want_test else None
    return tr, va, Xtr, Xva, Xte


_tr, _va, _Xtr, _Xva, _ = build_arm(PROBE_FOLD)
print(f"frame: {_Xtr.shape[1]} columns "
      f"({len(COLS) - len(CAT)} raw numeric + 24 encoded + {len(DUMMIES)} one-hot "
      f"+ {len(RATIO_COLS)} ratio)")
print(f"columns containing NaN: {int(_Xtr.isna().any().sum())} of {_Xtr.shape[1]}")
del _Xtr, _Xva
gc.collect()

## Stage 3. The four learners, and the preprocessing each one forces

`prep` is fit on training rows only, inside the fold, every time. `hgb` skips imputation
entirely because HistGradientBoosting routes NaN natively, which is the one place these arms
differ from each other by choice rather than by necessity.

In [ ]:
def make(arm, n_trees):
    if arm == "logit_te_fe":
        return LogisticRegression(C=1.0, max_iter=3000, solver="lbfgs")
    if arm == "hgb_te_fe":
        return HistGradientBoostingClassifier(
            learning_rate=LR, max_iter=n_trees, max_depth=MAX_DEPTH,
            min_samples_leaf=MIN_LEAF, l2_regularization=1.0,
            early_stopping=False, random_state=SEED)
    common = dict(n_estimators=n_trees, min_samples_leaf=MIN_LEAF, n_jobs=THREADS,
                  random_state=SEED, bootstrap=True)
    if arm == "et_te_fe":
        return ExtraTreesClassifier(**common)
    return RandomForestClassifier(**common)


def prep_fit(arm, Xtr):
    """Everything the learner forces, fit on training rows only."""
    if arm == "hgb_te_fe":
        return None, None                       # native NaN, no scaling
    med = np.nanmedian(Xtr.to_numpy(dtype=np.float64), axis=0)
    med = np.where(np.isnan(med), 0.0, med)     # a fully-NaN column would poison the fill
    if arm != "logit_te_fe":
        return med, None
    sc = StandardScaler().fit(np.where(np.isnan(Xtr.to_numpy(np.float64)), med,
                                       Xtr.to_numpy(np.float64)))
    return med, sc


def prep_apply(arm, A, med, sc):
    a = A.to_numpy(dtype=np.float64)
    if med is None:
        return a
    a = np.where(np.isnan(a), med, a)
    return sc.transform(a) if sc is not None else a


def hhmm(s):
    s = int(s)
    return f"{s // 3600}h {s % 3600 // 60:02d}m" if s >= 3600 else f"{s // 60}m {s % 60:02d}s"


ARM_TREES = {"logit_te_fe": 1, "hgb_te_fe": N_EST,
             "et_te_fe": N_TREES, "rf_te_fe": N_TREES}
BENCH_TREES = {"logit_te_fe": 1, "hgb_te_fe": BENCH_EST,
               "et_te_fe": max(10, N_TREES // 10), "rf_te_fe": max(10, N_TREES // 10)}

tr0, va0, Xtr0, Xva0, _ = build_arm(PROBE_FOLD)
bench = {}
for arm in ARMS:
    med, sc = prep_fit(arm, Xtr0)
    A, B = prep_apply(arm, Xtr0, med, sc), prep_apply(arm, Xva0, med, sc)
    t0 = time.time()
    m = make(arm, BENCH_TREES[arm]).fit(A, y[tr0])
    secs = time.time() - t0
    auc = roc_auc_score(y[va0], m.predict_proba(B)[:, 1])
    bench[arm] = secs
    print(f"{arm:14} bench: AUC {auc:.6f} in {hhmm(secs)} "
          f"({BENCH_TREES[arm]} units)")
    del m
    gc.collect()
del Xtr0, Xva0
gc.collect()

projected = sum(bench[a] / max(BENCH_TREES[a], 1) * ARM_TREES[a] * 5 for a in ARMS)
for a in ARMS:
    print(f"  {a:14} {hhmm(bench[a] / max(BENCH_TREES[a], 1) * ARM_TREES[a] * 5):>8}")
print(f"projected full run: {hhmm(projected)}")
GO = projected < MAX_HOURS * 3600 or SMOKE
print("within budget" if GO else f"OVER the {MAX_HOURS}h guard, not starting.")

## Stage 4. The run

In [ ]:
assert LEAK_OK and CLEAN, "leak checks failed"
assert ENCODER_MATCH, "encoder does not match 13"
assert BLOCK_OK, "the ratio block is not a pure function of the features"
assert GO, "over the time guard"
if not SMOKE:
    assert ALIGNED, "fold sha mismatch"

pre = "SMOKE_" if SMOKE else ""
results = {}
t_start = time.time()
for arm in ARMS:
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    per = []
    t_arm = time.time()
    for f in range(5):
        tr, va, Xtr, Xva, Xte = build_arm(f, want_test=True)
        med, sc = prep_fit(arm, Xtr)
        m = make(arm, ARM_TREES[arm]).fit(prep_apply(arm, Xtr, med, sc), y[tr])
        oof[va] = m.predict_proba(prep_apply(arm, Xva, med, sc))[:, 1]
        tst[f] = m.predict_proba(prep_apply(arm, Xte, med, sc))[:, 1]
        per.append(float(roc_auc_score(y[va], oof[va])))
        del m, Xtr, Xva, Xte
        gc.collect()
    results[arm] = {"oof": oof, "test": tst.mean(axis=0), "per": np.array(per)}
    np.save(OUT / f"{pre}{arm}_oof.npy", oof)
    np.save(OUT / f"{pre}{arm}_test.npy", tst.mean(axis=0))
    print(f"{arm:14} CV {np.mean(per):.6f} +/- {np.std(per):.6f}  "
          f"[{hhmm(time.time() - t_arm)}]")
print(f"\nall arms done in {hhmm(time.time() - t_start)}")

In [ ]:
print(f"{'arm':14}{'CV':>11}{'sd':>10}{'vs row 93':>12}   spearman vs xgb_te_fe")
try:
    ref = np.load(locate("xgb_te_fe_oof.npy"))
except FileNotFoundError:
    ref = None
for arm in ARMS:
    p = results[arm]["per"]
    rho = (pd.Series(results[arm]["oof"]).corr(pd.Series(ref[ROW_IDX]), method="spearman")
           if ref is not None else float("nan"))
    print(f"{arm:14}{p.mean():11.6f}{p.std():10.6f}{p.mean() - BASELINE_CV:+12.6f}"
          f"        {rho:.6f}")

print("\nDISAGREEMENT IS THE POINT, NOT ACCURACY. For scale, from earlier rows:")
print("  xgb_te vs its own seed twin        0.997344")
print("  cat_te_n4000 vs cat42              0.999208   (under the gate floor in row 99)")
print("  xgb_raw_fe vs xgb_te               0.983258   (FIRED at 5/5 folds, row 96)")
print("\nthis repo refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143,")
print("so a low number here is not a promise. The gate in 46 is the decision.")
if SMOKE:
    print("\nSMOKE: subsampled, none of these numbers resolve anything.")

In [ ]:
print("ledger lines:")
for arm in ARMS:
    p = results[arm]["per"]
    print(f"  name    {arm}\n  cv_mean {p.mean():.6f}\n  cv_std  {p.std():.6f}")
print(f"\n  leak checks {'PASS' if (LEAK_OK and CLEAN) else 'FAILED'}, "
      f"encoder {EXPECTED_ENCODER_FP if ENCODER_MATCH else 'MISMATCH'}, "
      f"ratio block pure {BLOCK_OK}, "
      f"fold alignment {'verified' if ALIGNED else 'MISMATCH'}")
print("\nNo submission csv. Membership is 46_stack_v3.ipynb and a separate ledger row.")